
# MarianMT (zh→en) — Baseline TF Training (Wuxia Domain)

**TFG Anonymous – Baseline NMT (MarianMT)**  
This notebook trains and evaluates to model **MarianMT** (`Helsinki-NLP/opus-mt-zh-in`) using to dataset of **wuxia** (Chinese->English) already preparado in format `datasets` (HF).





## 1) Preparation of the environment

In [ ]:

import os, random, math
import numpy as np

import torch
print("CUDA disponible:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Name of the GPU:", torch.cuda.get_device_name(0))



> **Requirements of the dataset**: directory HF Datasets with *splits* `train`, `validation`, `test` and columns `zh` (Chinese) and `en` (English):  
> `processed_data/wuxia_zh_en_clean/`

In [ ]:
# Configuration of carpetas for entorno LOCAL
from pathlib import Path
BASE_DIR = Path.cwd().parent.parent
BASE_DIR.mkdir(exist_ok=True)

# Structure repository
for sub in ["evaluation", "models", "processed_data"]:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Base:", BASE_DIR.resolve())
print("Structure created (if not existed):")
for p in ["evaluation", "models", "proccesed_data"]:
    print(" -", (BASE_DIR / p).resolve())



## 2) Configuration

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    # Paths (local)
    dataset_dir: Path  = BASE_DIR / "processed_data" / "wuxia_zh_en_clean"   # <- directory with dataset HuggingFace 
    output_dir: Path   = BASE_DIR / "models" / "marianmt_wuxia"              # <- here is saved the models
    ckpt_dir: Path     = BASE_DIR / "checkpoints"                          # <- checkpoints during training
    training_dir: Path = BASE_DIR / "training"         
    evaluation_dir: Path = BASE_DIR / "evaluation" 
    translate_dir: Path = BASE_DIR / "evaluation" / "translate"
    translate_file: Path =  "marianmt.txt"
    results_file: Path = "results.txt"
    # Columns of the dataset
    src_col: str = "zh"
    tgt_col: str = "en"

    # Model
    model_ckpt: str = "Helsinki-NLP/opus-mt-zh-en"

    # Training
    seed: int = 42
    max_source_length: int = 128
    max_target_length: int = 128
    batch_size: int = 16
    epochs: int = 10
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    early_stopping_patience: int = 3

    fraction: float = 1

cfg = Config()
print(cfg)


In [ ]:
import random, numpy as np, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)



# Semillas 
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
os.environ["PYTHONHASHSEED"] = str(cfg.seed)


if device.type == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)
    # For reproducibilidad estricta
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Semillas fijadas and backend configured.")


## 3) Load dataset (Hugging Face Datasets)

In [ ]:

from datasets import load_from_disk, DatasetDict

assert os.path.isdir(cfg.dataset_dir), f"The dataset was not found at: {cfg.dataset_dir}"
raw_ds: DatasetDict = load_from_disk(cfg.dataset_dir)
print(raw_ds)

# Validar columns
def _check_cols(ds, src_col, tgt_col, split):
    cols = ds.column_names
    assert src_col in cols and tgt_col in cols, f"El split '{split}' must contener columns '{src_col}' y '{tgt_col}'. Columns: {cols}"

for split in ["train", "validation", "test"]:
    assert split in raw_ds, f"Falta el split '{split}' en el dataset."
    _check_cols(raw_ds[split], cfg.src_col, cfg.tgt_col, split)

# tests 
def take_fraction(ds, frac, seed=42):
    if frac >= 1.0:
        return ds
    n = max(1, int(len(ds) * frac))
    return ds.shuffle(seed=seed).select(range(n))

train_ds = take_fraction(raw_ds["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(raw_ds["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(raw_ds["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[:2])
print(f"Tam. train/val/test (fraction={cfg.fraction}):", len(train_ds), len(val_ds), len(test_ds))


## 4) Load tokenizador and model MarianMT (zh->in)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(cfg.model_ckpt)


model = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_ckpt)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {cfg.model_ckpt}")
print(f"Parameters total: {n_params:,}")

## 5) Preprocesamiento and tokenization

In [ ]:
# tokenize examples
def preprocess_function(examples):
    # Tokenize texts source and target
    model_inputs = tokenizer(
        examples[cfg.src_col],
        max_length=cfg.max_source_length,
        padding=False,
        truncation=True
    )

    # Tokenize target with "labels"
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples[cfg.tgt_col],
            max_length=cfg.max_target_length,
            padding=False,
            truncation=True
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

#  tokenization to all the dataset
tokenized_datasets = raw_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_ds["train"].column_names
)

train_ds = take_fraction(tokenized_datasets["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(tokenized_datasets["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(tokenized_datasets["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[0])


## 6) Data collator

In [ ]:
from transformers import DataCollatorForSeq2Seq

# Is encarga of align dynamically the sequences and create batches listos for the model
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",   
    return_tensors="pt"
)

# Example of batch
batch = data_collator([train_ds[i] for i in range(2)])
for k, v in batch.items():
    print(f"{k}: shape={v.shape}, dtype={v.dtype}")

## 7) Configuration of training 



> **By default**          
> **Optimizador**: `AdamW` (with LR=2e-5, weight decay=0.01) of `Seq2SeqTrainer`   
> **Loss**: `CrossEntropyLoss` (token-level) of `AutoModelForSeq2SeqLM `





In [ ]:

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# Directory for results
run_dir = cfg.output_dir
run_dir.mkdir(parents=True, exist_ok=True)

# Arguments of training
training_args = Seq2SeqTrainingArguments(
    output_dir=str(run_dir),
    overwrite_output_dir=True,
    eval_strategy="epoch",                  # Evaluar to the final of each epoch
    save_strategy="epoch",                  # Save checkpoint by epoch
    save_total_limit=3,                      # Max number of checkpoints saved
    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    weight_decay=cfg.weight_decay,
    logging_dir=str(run_dir / "logs"),
    logging_strategy="steps",
    logging_steps=50,
    predict_with_generate=True,              # Generate sequences in validation
    fp16=torch.cuda.is_available(),          # Precision mixta if there are GPU
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

# Trainer for tareas Seq2Seq
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator
)

print(" Seq2SeqTrainer configured (PyTorch).")


## 8) Load model and tokenizador +  Charts

In [ ]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import gc

try:
    del model
    del tokenizer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

#best checkpoint saved
best_ckpt_path = getattr(trainer.state, "best_model_checkpoint", None)
if best_ckpt_path is None:
    # fallback
    best_ckpt_path = cfg.output_dir
    print("or is found 'best_model_checkpoint'. Loading from output_dir final.")

print(f"Loading model from: {best_ckpt_path}")

# Load model and tokenizer from the best checkpoint
tokenizer = AutoTokenizer.from_pretrained(best_ckpt_path)
model = AutoModelForSeq2SeqLM.from_pretrained(best_ckpt_path).to(device)

print("Best model and tokenizer cargados.")


In [ ]:

import json
import matplotlib.pyplot as plt
from pathlib import Path

#  Load info of the run 
run_info_path = Path(cfg.output_dir) / "run_info.json"
if run_info_path.exists():
    with open(run_info_path, "r", encoding="utf-8") as f:
        run_info = json.load(f)
    print(" Information of the training:")
    for k, v in run_info.items():
        print(f"  {k}: {v}")
else:
    print(f" Was not found {run_info_path}")

#   metrics final
metrics_path = Path(cfg.output_dir) / "train_results.json"
if metrics_path.exists():
    with open(metrics_path, "r", encoding="utf-8") as f:
        train_metrics = json.load(f)
    print("\n Metrics final of training:")
    for k, v in train_metrics.items():
        print(f"  {k}: {v}")
else:
    print(f" Was not found {metrics_path}")

#  trainer_state.json for history of training 
log_history_path = Path(cfg.output_dir) / "trainer_state.json"
if log_history_path.exists():
    with open(log_history_path, "r", encoding="utf-8") as f:
        trainer_state = json.load(f)

    # Info of the best checkpoint
    best_ckpt = trainer_state.get("best_model_checkpoint", None)
    if best_ckpt:
        print(f"\n Mejor checkpoint: {best_ckpt}")
    else:
        print("\n Was not found information of the best checkpoint.")

    # Historial of metrics
    log_history = trainer_state.get("log_history", [])

    # Extraer metrics and pasos
    steps_train = [entry["step"] for entry in log_history if "loss" in entry]
    train_loss = [entry["loss"] for entry in log_history if "loss" in entry]

    steps_eval = [entry["step"] for entry in log_history if "eval_loss" in entry]
    eval_loss = [entry["eval_loss"] for entry in log_history if "eval_loss" in entry]

    learning_rates = [entry["learning_rate"] for entry in log_history if "learning_rate" in entry]
    steps_lr = [entry["step"] for entry in log_history if "learning_rate" in entry]

    # Detectar metrics adicionales
    extra_metrics = {}
    for entry in log_history:
        for k, v in entry.items():
            if k.startswith("eval_") and k not in ["eval_loss"]:
                extra_metrics.setdefault(k, {"steps": [], "values": []})
                extra_metrics[k]["steps"].append(entry["step"])
                extra_metrics[k]["values"].append(v)

    #  Chart loss 
    plt.figure(figsize=(8,5))
    plt.plot(steps_train, train_loss, label="Train Loss")
    plt.plot(steps_eval, eval_loss, label="Eval Loss")
    plt.xlabel("Steps")
    plt.ylabel("Loss")
    plt.title("Evolution of the Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Chart learning rate
    if learning_rates:
        plt.figure(figsize=(8,5))
        plt.plot(steps_lr, learning_rates, label="Learning Rate", color="orange")
        plt.xlabel("Steps")
        plt.ylabel("LR")
        plt.title("Evolution of the rate of learning")
        plt.legend()
        plt.grid(True)
        plt.show()

    #  Charts of metrics adicionales 
    for metric_name, data in extra_metrics.items():
        plt.figure(figsize=(8,5))
        plt.plot(data["steps"], data["values"], label=metric_name)
        plt.xlabel("Steps")
        plt.ylabel(metric_name)
        plt.title(f"Evolution of {metric_name}")
        plt.legend()
        plt.grid(True)
        plt.show()

else:
    print(f"Was not found {log_history_path}")

## 9) Evaluation (SacreBLEU, chrF, TER, ROUGE-L, METEOR)

In [ ]:

from tqdm.auto import tqdm
import sacrebleu
from sacrebleu.metrics import CHRF, TER
from rouge_score import rouge_scorer
import nltk
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import wordpunct_tokenize
import numpy as np
import torch
import time


start = time.time()


# Descargar recursos of NLTK (for METEOR)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

#  Parameters 
EVAL_MAX_SAMPLES = None        # None = all the split
PRED_BEAMS = 4
BATCH_EVAL = max(1, cfg.batch_size // 2)


assert 'model' in globals(), "Was not found `model`. Loads the model before."
assert 'tokenizer' in globals(), "Was not found `tokenizer`. Load it before."
assert 'val_ds' in globals() and 'test_ds' in globals(), "Faltan `val_ds` y/o `test_ds`."
assert 'cfg' in globals(), "Falta `cfg`."

# Ensure pad_token_id
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id


#  Seleccionar split 
eval_raw = test_ds if len(test_ds) > 0 else val_ds
n_total = len(eval_raw)
n_eval = n_total if (EVAL_MAX_SAMPLES is None) else min(n_total, int(EVAL_MAX_SAMPLES))
assert n_eval > 0, "No there are examples for evaluar."
def decode_ids_to_text(dataset, id_col):
    return [
        tokenizer.decode(ids, skip_special_tokens=True)
        for ids in dataset[id_col]
    ]

src_texts = decode_ids_to_text(eval_raw, "input_ids")[:n_eval]
ref_texts = decode_ids_to_text(eval_raw, "labels")[:n_eval]


#  Generation by batches 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def batched_generate(texts, batch_size=8, max_length=128, num_beams=4):
    preds = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=cfg.max_source_length
            ).to(device)
            outputs = model.generate(
                **inputs,
                max_length=max_length,
                num_beams=num_beams,
                early_stopping=True
            )
            preds.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    return preds

preds = batched_generate(
    src_texts,
    batch_size=BATCH_EVAL,
    max_length=cfg.max_target_length,
    num_beams=PRED_BEAMS
)

#  Metrics 
bleu_corpus = sacrebleu.corpus_bleu(preds, [ref_texts]).score

chrf_metric = CHRF(word_order=2)
chrf_corpus = chrf_metric.corpus_score(preds, [ref_texts]).score

ter_metric = TER()
ter_corpus = ter_metric.corpus_score(preds, [ref_texts]).score

def compute_rougeL_f1(hyp_list, ref_list):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    f1s = []
    for h, r in zip(hyp_list, ref_list):
        s = scorer.score(r, h)
        f1s.append(s['rougeL'].fmeasure)
    return float(np.mean(f1s)) * 100.0
rougeL_f1 = compute_rougeL_f1(preds, ref_texts)

def compute_meteor(hyp_list, ref_list):
    scores = []
    for hyp, ref in zip(hyp_list, ref_list):
        hyp_tok = wordpunct_tokenize(hyp) if isinstance(hyp, str) else hyp
        ref_tok = wordpunct_tokenize(ref) if isinstance(ref, str) else ref
        scores.append(meteor_score([ref_tok], hyp_tok))
    return float(np.mean(scores)) * 100.0
meteor_avg = compute_meteor(preds, ref_texts)

end_time = time.time()

results = {
    "model" : cfg.model_ckpt,
    "n_eval": n_eval,
    "num_beams": PRED_BEAMS,
    "batch_eval": BATCH_EVAL,
    "sacrebleu": round(bleu_corpus, 4),
    "chrf2": round(chrf_corpus, 4),
    "ter": round(ter_corpus, 4),
    "rougeL_f1": round(rougeL_f1, 4),
    "meteor": round(meteor_avg, 4), 
    "execution_time": round(end_time - start, 2)
}

os.makedirs(cfg.evaluation_dir, exist_ok=True)

res_file = os.path.join(cfg.evaluation_dir, cfg.results_file)

with open(res_file, "a", encoding="utf-8") as f:
    f.write("\n")
    f.write(json.dumps(results, ensure_ascii=False, indent=4))

print(results)


64min

## 10) Sample cualitativa (n examples aleatorios)

In [ ]:

n_show = 100
idxs = random.sample(range(len(eval_raw)), k=min(n_show, len(eval_raw)))

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for i in idxs:
    # Decode text source and reference from dataset tokenized
    zh = tokenizer.decode(eval_raw[i]["input_ids"], skip_special_tokens=True)
    en_ref = tokenizer.decode(eval_raw[i]["labels"], skip_special_tokens=True)

    # Tokenize input and move to device
    inputs = tokenizer(zh, return_tensors="pt").to(device)

    # Generate translation
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_length=cfg.max_target_length,
            num_beams=4,
            early_stopping=True
        )

    en_pred = tokenizer.decode(out[0], skip_special_tokens=True)

    print("="*80)
    print("ZH:", zh)
    print("IN (ref):", en_ref)
    print("IN (pred):", en_pred)


In [ ]:
from tqdm import tqdm

os.makedirs(cfg.translate_dir, exist_ok=True)
translate_path = os.path.join(cfg.translate_dir, cfg.translate_file)

with open(translate_path, "w", encoding="utf-8") as f:
    for i in tqdm(range(len(eval_raw) // 2)):
        zh = tokenizer.decode(eval_raw[i]["input_ids"], skip_special_tokens=True)
        en_ref = tokenizer.decode(eval_raw[i]["labels"], skip_special_tokens=True)

        inputs = tokenizer(zh, return_tensors="pt").to(device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_length=cfg.max_target_length,
                num_beams=4,
                early_stopping=True
            )

        en_pred = tokenizer.decode(out[0], skip_special_tokens=True)

        # Save in the file
        f.write("="*80 + "\n")
        f.write("ZH: " + zh + "\n")
        f.write("IN (ref): " + en_ref + "\n")
        f.write("IN (pred): " + en_pred + "\n\n")


In [ ]:

# ==============================================================================
# CODE FOR EXTRAER METRICS OF DIFICULTAD (DATA CARTOGRAPHY)
# Add to the final of each notebook
# ==============================================================================

import torch
import pandas as pd
import sacrebleu
from tqdm.auto import tqdm
from sacrebleu.metrics import CHRF
# 1. Configuration
# ----------------
BATCH_SIZE = 64  # Increase it if have VRAM, decrease it if gives OOM
OUTPUT_FILE = f"scores_{cfg.model_ckpt.replace('/', '_')}.csv"
SRC_COL = cfg.src_col 
TGT_COL = cfg.tgt_col

print(f"--> Starting inference over the TRAIN SET for: {cfg.model_ckpt}")
print(f"--> Output file: {OUTPUT_FILE}")

# 2. Prepare Model
# ------------------
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Ensure configuration of language for models multilingual (mBART, M2M100)
if hasattr(tokenizer, "src_lang") and hasattr(cfg, "src_lang"):
    tokenizer.src_lang = cfg.src_lang
# For mBART/M2M100 forzamos the token of start if is necessary
if hasattr(model.config, "forced_bos_token_id") and hasattr(tokenizer, "lang_code_to_id"):
    try:
        # Intentar obtain the ID of the language target
        lang_id = tokenizer.lang_code_to_id.get(cfg.tgt_lang, None)
        if lang_id is not None:
            model.config.forced_bos_token_id = lang_id
    except:
        pass

# 3. Loop of Inference over Train
# ---------------------------------
# Use raw_ds['train'] for have the text crudo and calculate BLEU real
dataset = raw_ds['train'] 
# dataset = dataset.select(range(1000)) # DESCOMENTAR FOR PROBAR RAPIDO

results_indices = []
results_scores = []

# Iterate by batches
for i in tqdm(range(0, len(dataset), BATCH_SIZE), desc="Generating"):
    batch = dataset[i : i + BATCH_SIZE]
    src_texts = batch[SRC_COL]
    tgt_texts = batch[TGT_COL] # Referencias reales
    
    # to) Tokenize
    inputs = tokenizer(
        src_texts, 
        padding=True, 
        truncation=True, 
        max_length=cfg.max_source_length, 
        return_tensors="pt"
    ).to(device)
    
    # b) Generate (Greedy Search for speed)
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs, 
            max_new_tokens=cfg.max_target_length, 
            num_beams=1, 
            do_sample=False,
            early_stopping=False
        )
    
    # c) Decode
    decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    chrf_metric = CHRF(word_order=2)
    # d) Calcular Sentence-chrf
    for idx, (pred, ref) in enumerate(zip(decoded_preds, tgt_texts)):
        

        score = sacrebleu.sentence_chrf(pred, [ref]).score        
        # Save the index original of the dataset and the score
        results_indices.append(i + idx)
        results_scores.append(score)

# 4. Save Results
# ---------------------
df_scores = pd.DataFrame({
    'id': results_indices,
    'chrf': results_scores
})

# Save in the directory of processed_data or where prefieras
save_path = BASE_DIR / "processed_data" / OUTPUT_FILE
df_scores.to_csv(save_path, index=False)
print(f" Scores saved en: {save_path}")